## WEEK - 11 Multi-Agent RAG & Advanced Architectures

In [ ]:
import os
import json
from datetime import datetime
import numpy as np
import faiss

from google import genai
from sentence_transformers import SentenceTransformer

# ============================
# 🔑 GEMINI CLIENT (NEW SDK)
# ============================
# client = genai.Client(api_key="")

# ============================
# PROMPTS
# ============================
from prompts import (
    SYSTEM_PROMPT_AGENT_INGESTION,
    SYSTEM_PROMPT_AGENT_RETRIEVE,
    SYSTEM_PROMPT_AGENT_ANSWER,
    SYSTEM_PROMPT_AGENT_ROUTER,
)

DB_FILE = "local_db.txt"

# ============================
# EMBEDDING MODEL
# ============================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ============================
# GEMINI FUNCTION (NEW)
# ============================
def gemini_generate(system_prompt, user_prompt, retries=3):
    for i in range(retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=f"{system_prompt}\n\nUser:\n{user_prompt}"
            )
            return response.text.strip()

        except Exception as e:
            print(f"❌ Gemini Error (try {i}):", str(e))  # 🔥 IMPORTANT

    return "Error: Failed after retries"
   

# ============================
# DB LOAD
# ============================
def load_db():
    if not os.path.exists(DB_FILE):
        return []
    with open(DB_FILE, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]

# ============================
# INGESTION
# ============================
def text_to_db(text):
    with open(DB_FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")
    return "Stored successfully."

def path_to_db(path):
    if not os.path.exists(path):
        return "File not found."

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    with open(DB_FILE, "a", encoding="utf-8") as db:
        db.write(content + "\n")

    return "File ingested."

# ============================
# RETRIEVE (FAISS)
# ============================
def retrieve_str(query):
    data = load_db()
    if not data:
        return "No data available."

    doc_embeddings = embed_model.encode(data, convert_to_numpy=True)
    query_embedding = embed_model.encode([query], convert_to_numpy=True)

    dim = doc_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(doc_embeddings)

    D, I = index.search(query_embedding, k=min(5, len(data)))

    results = [data[i] for i in I[0]]

    print("\n🔎 Retrieved Context:\n", results, "\n")

    return "\n".join(results)

# ============================
# AGENTS
# ============================

def agent_router(user_query):
    response = gemini_generate(
        SYSTEM_PROMPT_AGENT_ROUTER,
        f"""
Decide:
- ingestion
- retrieve
- answer

Query: {user_query}

Only return one word.
"""
    ).lower()

    if "ingestion" in response:
        return "ingestion"
    elif "retrieve" in response:
        return "retrieve"
    else:
        return "answer"


def agent_ingestion(user_query):
    response = gemini_generate(
        SYSTEM_PROMPT_AGENT_INGESTION,
        user_query
    )

    if user_query.startswith("file:"):
        path = user_query.replace("file:", "").strip()
        tool_result = path_to_db(path)
    else:
        tool_result = text_to_db(user_query)

    return f"{response}\n\n[TOOL RESULT]: {tool_result}"


def agent_retrieve(user_query):
    context = retrieve_str(user_query)

    _ = gemini_generate(
        SYSTEM_PROMPT_AGENT_RETRIEVE,
        f"""
Query: {user_query}

Retrieved Context:
{context}
"""
    )

    return context


def agent_answer(context, user_query):
    response = gemini_generate(
        SYSTEM_PROMPT_AGENT_ANSWER,
        f"""
Context:
{context}

Question:
{user_query}
"""
    )
    return response

# ============================
# MAIN LOOP
# ============================
def run_chat():
    print("🔥 Multi-Agent System (Gemini NEW SDK + FAISS)\n")

    while True:
        query = input("You: ")

        if query.lower() == "exit":
            break

        route = agent_router(query)
        context = ""
        result = ""

        if route == "ingestion":
            result = agent_ingestion(query)

        elif route == "retrieve":
            context = agent_retrieve(query)
            result = agent_answer(context, query)

        else:
            result = agent_answer("", query)

        print(f"\nAssistant: {result}\n")


        save_log(
            query=query,
            route=route,
            context=context,
            response=result
        )
# ============================
# LOGGING
# ============================
def save_log(query, route, context, response):
    os.makedirs("chat_logs", exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    log_data = {
        "timestamp": timestamp,
        "query": query,
        "route": route,
        "retrieved_context": context,
        "response": response
    }

    with open(f"chat_logs/{timestamp}.json", "w") as f:
        json.dump(log_data, f, indent=4)

# ============================
# RUN
# ============================
if __name__ == "__main__":
    run_chat()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5651.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔥 Multi-Agent System (Gemini NEW SDK + FAISS)


Assistant: Hello! How can I help you today? Please let me know if you have any questions or if there's anything specific you'd like to know.

❌ Gemini Error (try 0): 'NoneType' object has no attribute 'strip'

Assistant: As the Answer Agent, I need the Retrieve Agent to provide me with information about the usage of Gemini. Once I have that information, I will answer your question based solely on what was retrieved.

Please provide the retrieved context so I can assist you further.


Assistant: The provided context does not contain information about "Gemini." Therefore, I am unable to answer your question.


🔎 Retrieved Context:
 ['my name is srutik i live in ahmedabad', 'what is my name?', 'my name is srutik nandnaiy ai have 4 cars', 'hi'] 


Assistant: I am the Answer Agent. My purpose is to answer your questions using information retrieved by the Retrieve Agent. I do not possess a knowledge base of my own.

Based on the information you

In [11]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain RAG in simple terms"
)

print(response.text)

Imagine you have a super smart friend who knows *a lot* of general information (that's your **Large Language Model** or **LLM**, like ChatGPT).

But this friend has two main limitations:

1.  **They have a cutoff date:** They only know things up until when they "read their last book." So, they might not know about very recent events or your company's latest internal policies.
2.  **They sometimes make things up (hallucinate):** If they don't know the exact answer, they might confidently give you a plausible-sounding but incorrect one.

Now, imagine you want to ask this super smart friend a question that requires very specific, up-to-date, or internal company knowledge.

**This is where RAG comes in!**

**RAG stands for "Retrieval Augmented Generation."** It's like giving your super smart friend a quick way to look up specific information *before* they answer your question.

Here's how it works in simple steps:

1.  **You Ask a Question:** You ask your super smart friend (the LLM) somet

In [4]:
from google import genai
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# List models
models = client.models.list()

for m in models:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [ ]:
n = int(input("n: "))
arr = []

while len(arr) < n:
    arr.extend(map(int, input("Enter elements: ").split()))

arr = arr[:n]
print(arr)

[1, 2, 3, 4, 5, 6, 7, 8]
[1, 2, 3]


In [17]:
b = []
for i in range(5):
    a = int(input())
    b.append(a)
print(b)

[3, 33, 3, 33, 33]


In [27]:
nin = ["hello","6,6,8,4","how","are","you"]
for i,n in enumerate(nin):
    print(i)
    print(n)

0
hello
1
6,6,8,4
2
how
3
are
4
you
